In [ ]:
# Cài đặt Unsloth trên Google Colab (Chỉ chạy 1 lần đầu phiên)
!pip install -q unsloth

import json
import os
from google.colab import drive
from unsloth import FastLanguageModel

# 1. Kết nối Google Drive
drive.mount('/content/drive')

LORA_DIR = "/content/drive/MyDrive/NovelAI/lora_adapter"
BASE_MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"

# Nếu đã có LoRA Adapter từ Giai đoạn 2 thì nạp LoRA, nếu chưa có thì dùng Base Model
active_model_path = LORA_DIR if os.path.exists(os.path.join(LORA_DIR, "adapter_config.json")) else BASE_MODEL_NAME
model_label = "Qwen3-4B-QLoRA" if active_model_path == LORA_DIR else "Qwen3-4B-Base-RAG"
print(f"Đang nạp mô hình từ: {active_model_path} ({model_label})")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = active_model_path,
    max_seq_length = 4096,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

INPUT_JSON_PATH = "/content/drive/MyDrive/NovelAI/input/pending_chunks.json"
OUTPUT_JSON_PATH = "/content/drive/MyDrive/NovelAI/output/translated_chunks.json"
os.makedirs(os.path.dirname(OUTPUT_JSON_PATH), exist_ok=True)

with open(INPUT_JSON_PATH, "r", encoding="utf-8") as f:
    pending_chunks = json.load(f)

# 2. Nạp danh sách các đoạn đã dịch (Checkpoint Resume)
completed_map = {}
if os.path.exists(OUTPUT_JSON_PATH):
    with open(OUTPUT_JSON_PATH, "r", encoding="utf-8") as f:
        for item in json.load(f):
            completed_map[item["chunk_id"]] = item

prev_translated_tail = ""
last_chapter_id = None

for item in pending_chunks:
    cid = item["chunk_id"]
    if cid in completed_map:
        prev_translated_tail = completed_map[cid]["output"][-250:]
        last_chapter_id = item["chapter_id"]
        print(f"[Đã dịch sẵn] Bỏ qua ChunkId={cid}")
        continue

    if item["chapter_id"] != last_chapter_id:
        prev_translated_tail = ""
        last_chapter_id = item["chapter_id"]

    ctx = item["context"]
    style_guide = ctx.get("style_guide", "")
    pronoun_rules = json.dumps(ctx.get("pronoun_rules", []), ensure_ascii=False, indent=2)
    few_shot = json.dumps(ctx.get("few_shot_examples", []), ensure_ascii=False, indent=2)

    system_prompt = (
        "Bạn là dịch giả tiểu thuyết chuyên nghiệp. Hãy dịch đoạn văn tiếng Anh ở phần 'input' sang tiếng Việt mượt mà, "
        "tuân thủ nghiêm ngặt quy tắc xưng hô trong 'pronoun_rules' và văn phong trong 'few_shot_examples'. "
        "Chỉ xuất ra bản dịch tiếng Việt, tuyệt đối không giải thích hay thêm ghi chú."
    )

    user_prompt = (
        f"### Hướng dẫn văn phong:\n{style_guide}\n\n"
        f"### Ngữ cảnh & Quy tắc xưng hô (SQL RAG):\n{pronoun_rules}\n\n"
        f"### Câu dịch mẫu đã duyệt:\n{few_shot}\n\n"
    )
    if prev_translated_tail:
        user_prompt += f"### Đoạn dịch ngay trước đó (Chỉ đọc để nối mạch văn, KHÔNG dịch lại):\n{prev_translated_tail}\n\n"
    user_prompt += f"### Văn bản tiếng Anh cần dịch (input):\n{item['input']}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = True
    )
    inputs = tokenizer([prompt_text], return_tensors="pt").to("cuda")
    input_token_count = int(inputs.input_ids.shape[1])

    outputs = model.generate(
        **inputs,
        max_new_tokens = 1500,
        temperature = 0.3,
        top_p = 0.9,
        use_cache = True
    )

    generated_ids = outputs[0][input_token_count:]
    output_token_count = int(len(generated_ids))
    vi_translation = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # 3. Đóng gói kết quả kèm số lượng Token để trả về bảng dbo.TranslationChunks
    completed_map[cid] = {
        "chunk_id": cid,
        "chapter_id": item["chapter_id"],
        "chapter_number": item["chapter_number"],
        "chunk_index": item["chunk_index"],
        "model_name": model_label,
        "input_tokens": input_token_count,
        "output_tokens": output_token_count,
        "output": vi_translation
    }

    # Ghi ngay xuống Google Drive sau mỗi đoạn
    with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as out_f:
        json.dump(list(completed_map.values()), out_f, ensure_ascii=False, indent=2)

    prev_translated_tail = vi_translation[-250:]
    print(f"Hoàn tất ChunkId={cid} (Đoạn {item['chunk_index']}) | Tokens: In={input_token_count}, Out={output_token_count}")